# Face detection and recognition training pipeline

The following example illustrates how to fine-tune an InceptionResnetV1 model on your own dataset. This will mostly follow standard pytorch training patterns.

In [2]:
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler
from torch import optim
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import numpy as np
import os

#### Define run parameters

The dataset follows the VGGFace2 directory layout. Modify `data_dir` to the location of the dataset on wish to finetune on.

In [3]:
data_dir = '../Image_dataset'

batch_size = 32
epochs = 8
workers = 0 if os.name == 'nt' else 8

#### Determine if an nvidia GPU is available

In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Running on device: {}'.format(device))

Running on device: cpu


#### Define MTCNN module

See `help(MTCNN)` for more details.

In [5]:
mtcnn = MTCNN(
    image_size=160, margin=0, min_face_size=20,
    thresholds=[0.6, 0.7, 0.7], factor=0.709, post_process=True,
    device=device
)

#### Perfom MTCNN facial detection

Iterate through the DataLoader object and obtain cropped faces.

In [6]:
dataset = datasets.ImageFolder(data_dir, transform=transforms.Resize((512, 512)))
dataset.samples = [
    (p, p.replace(data_dir, data_dir + '_cropped'))
        for p, _ in dataset.samples
]

loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=1,
    collate_fn=training.collate_pil
)


for i, (x, y) in enumerate(loader):
    mtcnn(x, save_path=y)
    print('\rBatch {} of {}'.format(i + 1, len(loader)), end='')
    
# Remove mtcnn to reduce GPU memory usage
del mtcnn

Batch 86055 of 86055

#### Define Inception Resnet V1 module

See `help(InceptionResnetV1)` for more details.

In [23]:
resnet = InceptionResnetV1(
    classify=True,
    pretrained='vggface2',
    # num_classes=len(dataset.class_to_idx) # Dòng code cũ khi chạy lỗi index out of box 235
    num_classes=len(dataset_cropped.class_to_idx)
).to(device)

#### Define optimizer, scheduler, dataset, and dataloader

In [24]:
# Tối ưu hóa và lịch trình giảm LR (change 1)
optimizer = optim.Adam(resnet.parameters(), lr=0.001)
scheduler = MultiStepLR(optimizer, milestones=[5, 10])

# Image Pre-processing
trans = transforms.Compose([
    transforms.Lambda(lambda x: np.array(x, dtype=np.float32)),
    transforms.ToTensor(),
    fixed_image_standardization
])

# Dataset và chia train/val
dataset_cropped = datasets.ImageFolder(data_dir + '_cropped', transform=trans)
# print(f"Number of classes in cropped dataset: {len(dataset_cropped.class_to_idx)}") # Added print for debugging
img_inds = np.arange(len(dataset_cropped))
np.random.shuffle(img_inds)
train_inds = img_inds[:int(0.8 * len(img_inds))]
val_inds = img_inds[int(0.8 * len(img_inds)):]

# DataLoaders
train_loader = DataLoader(
    dataset_cropped,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(train_inds)
)

val_loader = DataLoader(
    dataset_cropped,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(val_inds)
)

#### Define loss and evaluation functions

In [25]:
loss_fn = torch.nn.CrossEntropyLoss()
metrics = {
    'fps': training.BatchTimer(),
    'acc': training.accuracy
}

#### Train model

In [26]:
writer = SummaryWriter()
writer.iteration, writer.interval = 0, 10

print('\n\nInitial')
print('-' * 10)
resnet.eval()
training.pass_epoch(
    resnet, loss_fn, val_loader,
    batch_metrics=metrics, show_running=True, device=device,
    writer=writer
)

for epoch in range(epochs):
    print('\nEpoch {}/{}'.format(epoch + 1, epochs))
    print('-' * 10)

    resnet.train()
    training.pass_epoch(
        resnet, loss_fn, train_loader, optimizer, scheduler,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

    resnet.eval()
    training.pass_epoch(
        resnet, loss_fn, val_loader,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

writer.close()



Initial
----------
Valid |   539/539  | loss:    5.6365 | fps:   66.1955 | acc:    0.0017   

Epoch 1/50
----------
Train |  2155/2155 | loss:    2.3134 | fps:   18.2879 | acc:    0.4488   
Valid |   539/539  | loss:    1.2425 | fps:   59.5184 | acc:    0.6873   

Epoch 2/50
----------
Train |  2155/2155 | loss:    1.0137 | fps:   17.2231 | acc:    0.7393   
Valid |   539/539  | loss:    0.8412 | fps:   71.2757 | acc:    0.7836   

Epoch 3/50
----------
Train |  2155/2155 | loss:    0.6904 | fps:   22.6223 | acc:    0.8182   
Valid |   539/539  | loss:    0.6311 | fps:   58.7811 | acc:    0.8404   

Epoch 4/50
----------
Train |  2155/2155 | loss:    0.5116 | fps:   19.4741 | acc:    0.8629   
Valid |   539/539  | loss:    0.6252 | fps:   70.2826 | acc:    0.8414   

Epoch 5/50
----------
Train |  2155/2155 | loss:    0.3860 | fps:   22.2759 | acc:    0.8933   
Valid |   539/539  | loss:    0.5381 | fps:   72.1275 | acc:    0.8635   

Epoch 6/50
----------
Train |  2155/2155 | loss: 

KeyboardInterrupt: 